<a href="https://colab.research.google.com/github/sofia-seo-j/chantey_2026/blob/AIDM/Assignment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [79]:
import os
from collections import Counter

In [80]:
# Download Tiny Shakespeare
if not os.path.exists('input.txt'):
    os.system('wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt')

# Read the file
with open('input.txt', 'r', encoding='utf-8') as f:
    input_text = f.read()

# Sample text to encode
sample_text = """
To be, or not to be, that is the question:
Whether 'tis nobler in the mind to suffer
The slings and arrows of outrageous fortune,
Or to take arms against a sea of troubles
And by opposing end them. To die—to sleep,
No more; and by a sleep to say we end
The heart-ache and the thousand natural shocks
That flesh is heir to: 'tis a consummation
Devoutly to be wish'd. To die, to sleep;
To sleep, perchance to dream—ay, there's the rub:
For in that sleep of death what dreams may come,
When we have shuffled off this mortal coil,
Must give us pause—there's the respect
That makes calamity of so long life.
"""

# Assignments
Implement **BPE** and add 1000 tokens. Then compute the compression ratio and encode the sample text.

Report back the compression ration and the number of required tokens for *sample_text*.


---



## Preprocessing (Character Tokenization)

(b) Create a list of integers representing the text, where every unique character in the text is mapped to a unique integer ID (e.g., ’a’ → 1, ’b’ → 2)

In [81]:
# Letter to int id
chars = sorted(list(set(input_text)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
# Apply to the data
tokens = [stoi[ch] for ch in input_text]
# Example
print(type(tokens))
print(tokens[:20])
print("".join([itos[i] for i in tokens[:20]]))

<class 'list'>
[18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43, 52, 10, 0, 14, 43, 44, 53, 56]
First Citizen:
Befor


(c) Print the initial number of tokens in the sequence and the size of your initial vocabulary.

In [82]:
print("Initial sequence length:", len(tokens))
print("Initial vocabulary size:", len(chars))

Initial sequence length: 1115394
Initial vocabulary size: 65


## Byte Pair Encoding (BPE)

(a) Iteratively find the most frequent pair of adjacent token IDs in the sequence.

In [83]:
def get_pair_counts(token_sequence):
    pairs = Counter()
    for i in range(len(token_sequence) - 1):
        pair = (token_sequence[i], token_sequence[i+1])
        pairs[pair] += 1
    return pairs

pair_counts = get_pair_counts(tokens)

print("Most common pairs (int):")
print(pair_counts.most_common(5))

print("\nMost common pairs (chars):")
for (id1, id2), count in pair_counts.most_common(5):
    char1 = itos[id1]
    char2 = itos[id2]
    print(f"(('{char1}', '{char2}'), {count})")

Most common pairs (int):
[((43, 1), 27643), ((1, 58), 23837), ((58, 46), 22739), ((46, 43), 18203), ((58, 1), 16508)]

Most common pairs (chars):
(('e', ' '), 27643)
((' ', 't'), 23837)
(('t', 'h'), 22739)
(('h', 'e'), 18203)
(('t', ' '), 16508)


(b) Merge this pair into a new token ID (add it to your vocabulary).

In [84]:
def merge_pair(token_sequence, pair, new_token_id):
    new_sequence = []
    i = 0
    while i < len(token_sequence):
        # If current and next match the pair
        if i < len(token_sequence)-1 and (token_sequence[i], token_sequence[i+1]) == pair:
            new_sequence.append(new_token_id)
            i += 2  # Skip both tokens
        else:
            new_sequence.append(token_sequence[i])
            i += 1
    return new_sequence

(c) Replace all occurrences of the pair in the sequence with the new token ID.

In [85]:
num_merges = 1000
current_tokens = tokens.copy()
next_token_id = len(chars)
merge_rules = []

for i in range(num_merges):

    pair_counts = get_pair_counts(current_tokens)

    if not pair_counts:
        break

    best_pair = pair_counts.most_common(1)[0][0]

    merge_rules.append((best_pair, next_token_id))

    current_tokens = merge_pair(current_tokens, best_pair, next_token_id)

    next_token_id += 1

    if i % 100 == 0:
        print(f"Merge step {i}")

Merge step 0
Merge step 100
Merge step 200
Merge step 300
Merge step 400
Merge step 500
Merge step 600
Merge step 700
Merge step 800
Merge step 900


(d) Repeat for N = 1000 merges. Report the **compression ratio** (Initial sequence length / Final sequence length).


In [86]:
print("Initial length:", len(tokens))
print("Final length:", len(current_tokens))
print("Compression ratio:", len(tokens) / len(current_tokens))

Initial length: 1115394
Final length: 416705
Compression ratio: 2.6766993436603834


## Inference

Using the generated vocabulary, encode the sample text in the worksheet and
report the number tokens in the encoded sequence.

In [87]:
# Add UNK token (does NOT change previous tasks)
UNK_ID = len(stoi)

# Encode sample text safely
sample_tokens = [stoi.get(ch, UNK_ID) for ch in sample_text]

# sample_tokens = [stoi[ch] for ch in sample_text]
def encode_with_bpe(token_sequence, merge_rules):
    tokens = token_sequence.copy()

    for pair, new_token_id in merge_rules:
        tokens = merge_pair(tokens, pair, new_token_id)

    return tokens

encoded_sample = encode_with_bpe(sample_tokens, merge_rules)

print("Encoded sample length:", len(encoded_sample))

Encoded sample length: 225
